<a href="https://colab.research.google.com/github/SaiSanthosh1508/Foundation-Models-From-Scratch/blob/main/Transformer_from_Scratch.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import torch
import torch.nn as nn

# Architectural Components

## 1. Input Embeddings

In [ ]:
import math

class InputEmbeddings(nn.Module):

  def __init__(self, vocab_size : int, d_model : int):
    super().__init__()
    self.d_model = d_model
    self.embeddings = nn.Embedding(vocab_size,d_model)

  def forward(self, x):
    return self.embeddings(x) * math.sqrt(self.d_model)

## 2. Positional Encodings

In [ ]:
class PositionalEncodings(nn.Module):

  def __init__(self, d_model : int, seq_len : int, dropout : float):
    super().__init__()
    self.dropout = nn.Dropout(dropout)

    pe = torch.zeros(seq_len, d_model)

    positions = torch.arange(0,seq_len).unsqueeze(1)
    div_term = torch.pow(10000,torch.arange(0,d_model,2) / d_model)


    pe[:,0::2] = torch.sin(positions / div_term)
    pe[:,1::2] = torch.cos(positions / div_term)
    pe = pe.unsqueeze(0)

    self.register_buffer('pe',pe)

  def forward(self, x):
    return self.dropout(x + self.pe[:,:x.size(1)])

## 3. Multi Head Attention

In [ ]:
class MultiHeadAttention(nn.Module):

  def __init__(self, d_model : int, num_heads : int, dropout : float = 0.1):
    super().__init__()

    self.num_heads = num_heads
    self.d_model = d_model
    self.dropout = nn.Dropout(dropout)
    assert (d_model % num_heads) == 0, "d_model must be divisible by num_heads"

    self.W_q = nn.Linear(d_model, d_model)
    self.W_k = nn.Linear(d_model, d_model)
    self.W_v = nn.Linear(d_model, d_model)
    self.W_o = nn.Linear(d_model, d_model)

    self.d_head = d_model // num_heads


  def attention(self, q, k, v, padding_mask=None):
    d_head = q.shape[-1]

    scaled_dot_product = torch.matmul(q,k.transpose(-1,-2)) / math.sqrt(d_head)

    if padding_mask is not None:
      scaled_dot_product = scaled_dot_product.masked_fill(padding_mask==0,float('-inf'))

    attn_weights = nn.Softmax(dim=-1)(scaled_dot_product)

    if self.dropout:
      attn_weights = self.dropout(attn_weights)

    return torch.matmul(attn_weights,v)

  def forward(self, q_input,k_input,v_input, padding_mask=None):
    B,S,_ = q_input.shape

    q = self.W_q(q_input)
    k = self.W_k(k_input)
    v = self.W_v(v_input)

    q = q.view(B,q.shape[1],self.num_heads,self.d_head).transpose(1,2)
    k = k.view(B,k.shape[1],self.num_heads,self.d_head).transpose(1,2)
    v = v.view(B,v.shape[1],self.num_heads,self.d_head).transpose(1,2)

    attn_weights = self.attention(q,k,v,padding_mask)

    return self.W_o(attn_weights.transpose(1,2).contiguous().view(B,S,-1))

## 4. Feed-Forward Block

In [ ]:
import torch.nn as nn
import torch

class FeedForwardBlock(nn.Module):
    def __init__(self, d_model : int, d_ff : int, dropout: float = 0.1):
        super().__init__()

        self.d_model = d_model
        self.d_ff = d_ff

        self.linear1 = nn.Linear(d_model,d_ff)
        self.linear2 = nn.Linear(d_ff,d_model)

        self.dropout = nn.Dropout(dropout)

    def forward(self, x):

        x = self.linear2(self.dropout(torch.relu(self.linear1(x))))
        return x

## 5. Residual Connections

In [ ]:
class ResidualConnections(nn.Module):

  def __init__(self, d_model : int, dropout : float):
    super().__init__()
    self.dropout = nn.Dropout(dropout)
    self.norm = nn.LayerNorm(d_model)

  def forward(self, x, sublayer):
    return self.norm(x + self.dropout(sublayer(x)))

## 6. Encoder Block

In [ ]:
class EncoderBlock(nn.Module):

  def __init__(self, d_model : int, d_ff : int, num_heads : int, dropout : float = 0.1):
    super().__init__()

    self.d_model = d_model
    self.d_ff = d_ff
    self.dropout = nn.Dropout(dropout)

    self.attention = MultiHeadAttention(d_model,num_heads,dropout)
    self.residuals = nn.ModuleList([ResidualConnections(d_model,dropout) for _ in range(2)])
    self.ffn = FeedForwardBlock(d_model,d_ff,dropout)

  def forward(self, x, padding_mask=None):
    attn = self.attention
    output = self.residuals[0](x, lambda x : attn(x,x,x,padding_mask))

    ffn = self.ffn
    output = self.residuals[1](output, self.ffn)

    return output

## Encoder

In [ ]:
class Encoder(nn.Module):

  def __init__(self, d_model : int, d_ff : int, num_heads : int, N : int, dropout : float = 0.1):
    super().__init__()

    self.encoders = nn.ModuleList([
        EncoderBlock(d_model,d_ff,num_heads,dropout) for _ in range(N)
    ])

  def forward(self,x, padding_mask=None):
    for layer in self.encoders:
      x = layer(x,padding_mask)

    return x

## 7. Decoder Block

In [ ]:
class DecoderBlock(nn.Module):
    def __init__(self, d_model : int, d_ff : int, num_heads : int, dropout : float = 0.1):
        super().__init__()

        self.masked_mha = MultiHeadAttention(d_model, num_heads, dropout)
        self.cross_mha = MultiHeadAttention(d_model, num_heads, dropout)
        self.ffn = FeedForwardBlock(d_model, d_ff, dropout)

        self.residuals = nn.ModuleList([
            ResidualConnections(d_model, dropout) for _ in range(3)
        ])

    def forward(self, x, encoder_output, src_mask, tgt_mask):

      res0,res1,res2 = self.residuals[0],self.residuals[1],self.residuals[2]

      masked_attn_out = res0(x, lambda x: self.masked_mha(x, x, x, tgt_mask))

      cross_attn_out = res1(masked_attn_out, lambda x: self.cross_mha(x, encoder_output, encoder_output, src_mask))

      ffn_out = res2(cross_attn_out,self.ffn)

      return ffn_out

## Decoder

In [ ]:
import torch.nn as nn

class Decoder(nn.Module):
    def __init__(self, d_model : int, d_ff : int, num_heads : int, N : int, dropout : float = 0.1):
        super().__init__()

        self.decoders = nn.ModuleList([
             DecoderBlock(d_model, d_ff, num_heads, dropout) for _ in range(N)
        ])

    def forward(self, x, encoder_output, src_mask, tgt_mask):

        for layer in self.decoders:
            x = layer(x, encoder_output, src_mask, tgt_mask)

        return x

# Task : English to French Translation

## Transformer Architecture For Text Translation

In [ ]:
import torch.nn as nn

class TransformerForTranslation(nn.Module):

    def __init__(self, src_vocab_size: int, tgt_vocab_size: int, d_model: int, d_ff: int,
                 num_heads: int, N: int, seq_len: int, dropout: float = 0.1):
        super().__init__()

        self.src_embeddings = InputEmbeddings(src_vocab_size, d_model)
        self.tgt_embeddings = InputEmbeddings(tgt_vocab_size, d_model)

        self.pe = PositionalEncodings(d_model, seq_len, dropout)

        self.encoder = Encoder(d_model, d_ff, num_heads, N, dropout)
        self.decoder = Decoder(d_model, d_ff, num_heads, N, dropout)

        self.linear = nn.Linear(d_model, tgt_vocab_size)


    def forward(self, src, tgt, src_mask, tgt_mask):

        src_embedded = self.src_embeddings(src)

        src_input = self.pe(src_embedded)

        encoder_output = self.encoder(src_input, src_mask)

        tgt_embedded = self.tgt_embeddings(tgt)
        tgt_input = self.pe(tgt_embedded)
        decoder_output = self.decoder(tgt_input, encoder_output, src_mask, tgt_mask)

        logits = self.linear(decoder_output)

        return logits

In [ ]:
!pip install -q transformers datasets sacremoses

## Data Preprocessing

In [ ]:
from datasets import load_dataset

ds = load_dataset("LT3/nfr_bt_nmt_english-french",split="train[:6000]")
ds

In [ ]:
ds = ds.train_test_split(test_size=0.3)
ds

In [ ]:
ds['train'][10]

In [ ]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("Helsinki-NLP/opus-mt-en-fr")
tokenizer

In [ ]:
tokenizer.all_special_tokens

In [ ]:
PAD_TOKEN = tokenizer.pad_token
EOS_TOKEN = tokenizer.eos_token

PAD_TOKEN, EOS_TOKEN

In [ ]:
BOS_TOKEN = "<s>"
tokenizer.add_special_tokens({
    "bos_token" : BOS_TOKEN
})

tokenizer.all_special_tokens,len(tokenizer)

In [ ]:
from transformers import AutoTokenizer

def preprocess_fn(examples, tokenizer, MAX_SEQ_LEN):

    PAD_ID = tokenizer.pad_token_id
    SOS_ID = tokenizer.bos_token_id
    EOS_ID = tokenizer.eos_token_id

    processed_examples = {
        "src_ids": [],
        "src_attn_mask": [],
        "tgt_input_ids": [],
        "labels": [],
    }

    def pad_sequence(ids, length):
        return ids + [PAD_ID] * (length - len(ids))

    for eng, fre in zip(examples['english'], examples['french']):

        src_tokenized = tokenizer(
            eng,
            max_length=MAX_SEQ_LEN,
            padding="max_length",
            truncation=True,
        )

        tgt_tokens = tokenizer.encode(fre, max_length=MAX_SEQ_LEN - 1, truncation=True, add_special_tokens=False)

        # Decoder Input: [SOS] + Target tokens + [Padding]
        tgt_input_ids = [SOS_ID] + tgt_tokens
        tgt_input_padded = pad_sequence(tgt_input_ids, MAX_SEQ_LEN)

        # Labels: Target tokens + [EOS] + [Padding]
        labels_ids = tgt_tokens + [EOS_ID]
        labels_padded = pad_sequence(labels_ids, MAX_SEQ_LEN)

        processed_examples['src_ids'].append(src_tokenized['input_ids'])
        processed_examples['src_attn_mask'].append(src_tokenized['attention_mask'])
        processed_examples['tgt_input_ids'].append(tgt_input_padded)
        processed_examples['labels'].append(labels_padded)

    return processed_examples

In [ ]:

processed_ds = ds.map(preprocess_fn,batched=True,remove_columns=ds['train'].column_names,
                      fn_kwargs={'tokenizer': tokenizer, 'MAX_SEQ_LEN': 512})
processed_ds

In [ ]:
processed_ds.set_format(type="torch")

In [ ]:
from torch.utils.data import DataLoader

train_dl = DataLoader(processed_ds['train'],batch_size=8,shuffle=True)
test_dl = DataLoader(processed_ds['test'],batch_size=8,shuffle=False)

len(train_dl),len(test_dl)

In [ ]:
batch = next(iter(train_dl))

print(f"src_ids shape (Encoder Input): {batch['src_ids'].shape}")
print(f"tgt_input_ids shape (Decoder Input): {batch['tgt_input_ids'].shape}")
print(f"labels shape (Ground Truth): {batch['labels'].shape}")

## Model Training & Evaluation

In [ ]:
from torch.optim import AdamW

translation_model = TransformerForTranslation(
    src_vocab_size = len(tokenizer),
    tgt_vocab_size = len(tokenizer),
    d_model = 512,
    d_ff = 2048,
    num_heads = 8,
    N = 8,
    seq_len = 512,
    dropout = 0.2
)

optimizer = AdamW(translation_model.parameters(),lr=1e-4)

loss_fn = nn.CrossEntropyLoss(ignore_index=tokenizer.pad_token_id)

In [ ]:
import torch

def create_masks(src_input_ids, tgt_input_ids, pad_token_id):

    device = src_input_ids.device

    # 1. Source Mask (Padding Mask only) [B, 1, 1, S_src]
    src_mask = (src_input_ids != pad_token_id).unsqueeze(1).unsqueeze(2)

    # 2. Target Padding Mask [B, 1, 1, S_tgt]
    tgt_pad_mask = (tgt_input_ids != pad_token_id).unsqueeze(1).unsqueeze(2)

    # 3. Target Causal Mask (Look-Ahead Mask)
    tgt_seq_len = tgt_input_ids.shape[1]

    tgt_causal_mask = torch.tril(
        torch.ones(tgt_seq_len, tgt_seq_len, device=device)
    ).type(torch.bool)

    # Expand to [1, 1, S_tgt, S_tgt] for broadcasting
    tgt_causal_mask = tgt_causal_mask.unsqueeze(0).unsqueeze(0)

    # 4. Combined Target Mask (Padding AND Causal)
    tgt_mask = tgt_pad_mask & tgt_causal_mask

    return src_mask, tgt_mask

In [ ]:
import torch

# Check for CUDA device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

# Move your model to the selected device
translation_model.to(device)

In [ ]:
from tqdm import tqdm

def train_epoch(model, dataloader, optimizer, criterion, pad_idx, device):

    model.train() # Set the model to training mode
    total_loss = 0
    loop = tqdm(dataloader,desc="[Training]",leave=False)

    for batch in loop:
        # 0. Move data to device
        src_ids = batch['src_ids'].to(device)
        tgt_input_ids = batch['tgt_input_ids'].to(device)
        labels = batch['labels'].to(device)

        src_mask, tgt_mask = create_masks(src_ids, tgt_input_ids, pad_idx)

        optimizer.zero_grad()

        logits = model(src_ids, tgt_input_ids, src_mask, tgt_mask)

        B, S, V = logits.shape

        loss = criterion(logits.view(B * S, V), labels.view(B * S))

        loss.backward()

        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)

        optimizer.step()

        total_loss += loss.item()
        loop.set_postfix(loss=loss.item())

    return total_loss / len(dataloader)

In [ ]:
import torch
from tqdm.auto import tqdm

@torch.no_grad()
def validate_epoch(model, dataloader, criterion, pad_idx, device):
    model.eval()
    total_loss = 0

    loop = tqdm(dataloader, desc="Validation", leave=False)

    for batch in loop:
        src_ids = batch['src_ids'].to(device)
        tgt_input_ids = batch['tgt_input_ids'].to(device)
        labels = batch['labels'].to(device)

        src_mask, tgt_mask = create_masks(src_ids, tgt_input_ids, pad_idx)

        logits = model(src_ids, tgt_input_ids, src_mask, tgt_mask)

        B, S, V = logits.shape
        loss = criterion(logits.view(B * S, V), labels.view(B * S))

        total_loss += loss.item()

        loop.set_postfix(loss=loss.item())

    return total_loss / len(dataloader)

In [ ]:
N_EPOCHS = 3
PAD_IDX = tokenizer.pad_token_id

print("Starting training...")

for epoch in range(1, N_EPOCHS + 1):
    print(f"Epoch: {epoch}\n")
    train_loss = train_epoch(translation_model, train_dl, optimizer, loss_fn, PAD_IDX, device)

    val_loss = validate_epoch(translation_model, test_dl, loss_fn, PAD_IDX, device)

    print(f"Epoch {epoch}/{N_EPOCHS}:")
    print(f"  --> Train Loss: {train_loss:.4f}")
    print(f"  --> Validation Loss: {val_loss:.4f}")

## Inference

In [ ]:
import torch
import torch.nn.functional as F

@torch.no_grad()
def greedy_decode(model, input_text: str, tokenizer, device, max_new_tokens: int):

    model.eval()
    MAX_SEQ_LEN_ENCODER = 512

    src_encoding = tokenizer(
        input_text, max_length=MAX_SEQ_LEN_ENCODER, truncation=True, padding="max_length", return_tensors="pt"
    )
    src_ids = src_encoding['input_ids'].to(device)
    src_mask = (src_ids != tokenizer.pad_token_id).unsqueeze(1).unsqueeze(2).to(device)

    src_input = model.pe(model.src_embeddings(src_ids))
    encoder_output = model.encoder(src_input, src_mask)

    tgt_input_ids = torch.tensor([[tokenizer.bos_token_id]], dtype=torch.long, device=device)

    for _ in range(max_new_tokens):

        S_curr = tgt_input_ids.size(1)

        tgt_pad_mask = (tgt_input_ids != tokenizer.pad_token_id).unsqueeze(1).unsqueeze(2)
        tgt_causal_mask = torch.tril(torch.ones(S_curr, S_curr, device=device)).type(torch.bool)
        tgt_causal_mask = tgt_causal_mask.unsqueeze(0).unsqueeze(0) # Shape: [1, 1, S_curr, S_curr]
        tgt_mask = tgt_pad_mask & tgt_causal_mask

        tgt_input = model.pe(model.tgt_embeddings(tgt_input_ids)) # Shape: [1, S_curr, D]

        decoder_output = model.decoder(tgt_input, encoder_output, src_mask, tgt_mask) # Shape: [1, S_curr, D]

        last_decoder_output = decoder_output[:, -1, :] # Shape: [1, D]

        next_token_logits = model.linear(last_decoder_output) # Shape: [1, V]

        next_word_id = torch.argmax(next_token_logits, dim=-1).item() # Scalar (int)

        new_token = torch.tensor([[next_word_id]], dtype=torch.long, device=device) # Shape: [1, 1]

        tgt_input_ids = torch.cat([tgt_input_ids, new_token], dim=1) # Shape: [1, S_curr + 1]

        if next_word_id == tokenizer.eos_token_id:
            break

    # 5. Final Output
    generated_ids = tgt_input_ids.squeeze(0).tolist()
    return tokenizer.decode(generated_ids, skip_special_tokens=True)

In [ ]:
response = greedy_decode(translation_model,ds['test'][10]['english'],tokenizer,device,max_new_tokens=50)

In [ ]:
print("English text: ",ds['test'][10]['english'])
print("French translation:", response)